# Лабораторная работа: матричный autograd и многослойный перцептрон

Цель — пройти полный цикл CRISP-DM: понять задачу Wine, проверить и подготовить данные, реализовать `MatrixValue`, собрать MLP произвольной глубины, обучить её mini-batch SGD, оценить независимый test и сохранить модель.

В этой заготовке 18 исполнимых ячеек. Первые 14 заполняйте с нуля по полным листингам PDF лабораторной: после каждой ячейки разберите результат её проверок. `MatrixValue` вводится один раз полным классом; затем отдельные ячейки проверяют его операции.

После `single_batch_backward` разберите все ячейки выполненного `dry_bean_mlp_full_example.ipynb`. Его прикладной код послужит образцом для заполнения ячеек 15–18 Wine; точные изменения и полные Wine-листинги приведены в лабораторной. Запускайте свою тетрадь строго сверху вниз: каждая следующая ячейка использует определения предыдущих. Код документируется русскими Google-style docstring с английскими секциями `Args`, `Returns`, `Raises` и `Notes`.

Обозначения сохраняются во всех ячейках. Полный датасет — матрица $\mathbf X^{\mathrm{data}}\in\mathbb R^{n\times d}$; объект $i$ — вектор $\mathbf x_i$; число $x_{ij}$ — его признак $j$. Batch с индексами $\mathcal I=(i_1,\ldots,i_B)$ — матрица $\mathbf X_{\mathcal I}\in\mathbb R^{B\times d}$. Вход MLP обозначается $\mathbf H_{\mathcal I}^{(-1)}=\mathbf X_{\mathcal I}$.

> **Интуитивно: batch.** Это несколько объектов, обрабатываемых одной матричной операцией; один объект занимает одну строку.

В покомпонентных формулах $i$ — индекс строки-объекта, $j$ — входной координаты, $s$ — нейрона, $c$ — класса. Буква $h$ обозначает значение активации и не используется как индекс: запись имеет вид $h_{is}^{(\ell)}$. Немыми индексами явных сумм являются греческие буквы.

Ожидаемый итог: train accuracy `1.0`, validation/test accuracy не ниже `0.95`, конечная test CCE меньше `0.2`, сохранённый `wine_mlp_model.npz` воспроизводит прогнозы.

## Ячейка 1. Business Understanding и численные основы

Заполняется блок `imports_and_helpers`.

Три класса требуют вероятностного выхода по строкам. Сначала проверяется устойчивый softmax; это ещё не обучающий узел графа. Вычитание максимального логита из каждой строки не меняет отношение вероятностей, но предотвращает переполнение `exp`.

**Действие.** Перенесите ячейку импортов и функцию `softmax_rows`, затем запустите проверки экстремальных логитов.

**Контроль.** Полученная матрица должна состоять из конечных чисел, а сумма каждой строки должна быть равна `1.0`.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: imports_and_helpers
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 2. Data Understanding: локальный Wine

Заполняется блок `load_wine`.

Набор загружается без сети. Одна строка соответствует одному химическому образцу вина; тринадцать столбцов содержат числовые признаки, а метка выбирает один из трёх классов.

**Действие.** Выполните загрузку локального `data/wine.csv` и не изменяйте порядок признаков.

**Контроль.** Матрица признаков имеет форму `(178, 13)`, вектор меток — `(178,)`, частоты классов равны `59`, `71` и `48`; все признаки конечны.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: load_wine
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 3. Data Preparation

Заполняется блок `split_and_standardize`.

Стратифицированное разбиение сохраняет пропорции классов с точностью до целочисленного округления. Размеры частей — 125 / 27 / 26. Внутри каждой части места распределяются методом наибольших остатков: идеальные квоты пропорциональны исходным численностям 59 / 71 / 48. Получаются train 41 / 50 / 34, validation 9 / 11 / 7, test 9 / 10 / 7. Строки каждого класса перемешиваются и распределяются без пересечений; равных квот разных классов здесь нет. Сначала данные разделяются, и только затем mean и std вычисляются по train. Иначе validation и test заранее повлияли бы на подготовку обучающих данных.

Разбиение вызывается один раз с `seed=42`. Состав validation после этого
фиксирован на все эпохи; перемешивается только порядок train-объектов.

Для каждого признака используется преобразование $x^{\mathrm{std}}=(x-\bar x_{\mathrm{train}})/s_{\mathrm{train}}$. Те же train-статистики применяются к validation и test.

**Действие.** Создайте части `125 / 27 / 26`, сохраните сырые test-признаки и стандартизируйте все части.

**Контроль.** Только train обязан иметь среднее `0` и стандартное отклонение `1` с машинной точностью.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: split_and_standardize
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 4. Modeling: вершина матричного DAG

Заполняется блок `matrix_value`.

`MatrixValue` переносит принципы скалярного reverse-mode: `data`, `grad`, `parents`, локальный callback и обратный топологический обход. Один узел хранит целую матрицу.

> **Интуитивно: callback.** Операция сохраняет при forward локальную инструкцию `_backward`, а выполняет её позже, когда влияние результата на итоговую потерю уже известно.

Инвариант: непосредственно перед `Q._backward()` поле `Q.grad` уже равно полному входящему градиенту $\mathbf G_{\mathbf Q}=\nabla_{\mathbf Q}L$. Callback только преобразует его в локальные вклады родителей.

Если `Q.data` содержит элементы $q_{ij}$, то элемент `Q.grad[i, j]` равен частной производной $\partial L/\partial q_{ij}$. Поле `grad` имеет ту же форму, чтобы каждому числу прямого прохода соответствовал собственный коэффициент чувствительности потери.

**Действие.** Целиком перенесите один полный листинг класса из PDF в `matrix_value` и выполните ячейку. Затем переходите к отдельным проверкам операций. Не заменяйте `+=` на `=` и не меняйте `data` до завершённого `backward()`.

**Контроль.** Нульмерная потеря принимает неявный seed `1`; матрица даже формы `(1, 1)` требует явный seed той же формы.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: matrix_value
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 5. Транспонирование

Заполняется блок `transpose_trace`.

Если $\mathbf Q=\mathbf A^{\mathsf T}$, то каждый элемент только меняет местами координаты. Поэтому обратный вклад равен $\mathbf G_{\mathbf A}\mathrel{+}=\mathbf G_{\mathbf Q}^{\mathsf T}$.

**Действие.** Передайте результату seed формы `(3, 2)`.

**Контроль.** Родитель формы `(2, 3)` должен получить точно `seed.T`.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: transpose_trace
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 6. Матричное произведение как линейная часть слоя

Заполняется блок `matmul_trace`.

Каждый слой вычисляет

$$
\mathbf R^{(\ell)}
=
\tilde{\mathbf H}^{(\ell-1)}
(\mathbf W^{(\ell)})^{\mathsf T}.
$$

Поэтому проверка операции `@` подтверждает сразу две части backpropagation слоя: передачу градиента к предыдущим активациям и вычисление градиента весов.

Для общего произведения $\mathbf M=\mathbf A\mathbf B$ callback реализует
$\mathbf G_{\mathbf A}\mathrel{+}=\mathbf G_{\mathbf M}\mathbf B^{\mathsf T}$ и
$\mathbf G_{\mathbf B}\mathrel{+}=\mathbf A^{\mathsf T}\mathbf G_{\mathbf M}$.
В слое этим буквам соответствуют $\mathbf A=\tilde{\mathbf H}^{(\ell-1)}$,
$\mathbf B=(\mathbf W^{(\ell)})^{\mathsf T}$ и $\mathbf M=\mathbf R^{(\ell)}$.
Обе операции используют `+=`, потому что родитель может участвовать и в других путях графа.

**Действие.** Проверьте обычное произведение, затем отдельный граф `shared @ shared`, где один объект является обоими родителями результата.

**Контроль.** Для обычного произведения совпадают обе матричные формулы; для общего родителя его `grad` содержит сумму двух вкладов, а не последний вклад.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: matmul_trace
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 7. Постоянная координата

Заполняется блок `augment_trace`.

Родительская $\mathbf H\in\mathbb R^{B\times m}$ является матрицей: её строки соответствуют объектам batch, а столбцы — значениям предыдущего уровня. Forward создаёт матрицу-столбец $\mathbf1_B\in\mathbb R^{B\times1}$ и присоединяет её слева по оси столбцов:

$$
\tilde{\mathbf H}
=
[\mathbf1_B\ \mathbf H]
\in\mathbb R^{B\times(m+1)}.
$$

Backward удаляет градиент константной колонки и передаёт родительской матрице только $(\mathbf G_{\tilde{\mathbf H}})_{:,1:}$.

**Действие.** Создайте расширенную матрицу, передайте seed с намеренно крупными числами в первой колонке.

**Контроль.** Эти числа не должны попасть в `hidden.grad`; форма родительского градиента остаётся `(B, m)`.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: augment_trace
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 8. ReLU и tanh

Заполняется блок `activation_trace`.

Предактивация — сырой линейный ответ нейрона до нелинейности. Активация — значение после ReLU или tanh, которое передаётся следующему слою.

Обе активации работают поэлементно и сохраняют форму матрицы. Их VJP умножает входящий градиент на локальную производную: индикатор $\mathbb I[\mathbf R>0]$ для ReLU и $1-\tanh^2(\mathbf R)$ для tanh.

**Действие.** Выполните два свежих графа — отдельно для ReLU и tanh.

**Контроль.** ReLU пропускает градиент только при положительной предактивации; градиент tanh совпадает с $1-h^2$.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: activation_trace
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 9. Fused softmax–CCE

Заполняется блок `softmax_cross_entropy_trace`.

Логит — сырой балл класса, а не вероятность. Softmax превращает строку логитов в вероятности с суммой $1$. Categorical cross-entropy штрафует модель отрицательным логарифмом вероятности правильного класса.

Устойчивый forward вычитает максимум каждой строки. После применения цепного правила к softmax и categorical cross-entropy callback получает
$\mathbf G_{\mathbf Z}=(\mathbf P-\mathbf Y)/B$. Результат является скалярной средней потерей, поэтому `backward()` использует seed $1$.

**Действие.** Постройте loss для двух строк с экстремальными, но правильными логитами и вызовите `backward()`.

**Контроль.** Потеря и градиент конечны; сумма градиентов каждой строки логитов равна нулю.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: softmax_cross_entropy_trace
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 10. Независимая численная проверка

Заполняется блок `matrix_finite_difference`.

Центральная разность оценивает производную двумя дополнительными forward при $q+\varepsilon$ и $q-\varepsilon$. Она не участвует в обучении и служит независимой проверкой callbacks.

**Действие.** Сравните аналитический `grad` с численной оценкой для одного элемента матричного множителя и одного логита fused loss.

**Контроль.** Обе пары значений должны совпасть в установленном допуске.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: matrix_finite_difference
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 11. Общая MLP

Заполняется блок `mlp_classifier`.

Пусть сеть имеет $K$ скрытых слоёв. Вход имеет служебный индекс $-1$, скрытые слои — индексы $0,\ldots,K-1$, выходной слой — индекс $K$. Поэтому

$$
\mathbf H_{\mathcal I}^{(-1)}=\mathbf X_{\mathcal I},
\qquad
\mathbf R^{(\ell)}=\tilde{\mathbf H}^{(\ell-1)}(\mathbf W^{(\ell)})^{\mathsf T},
\qquad 0\leq\ell\leq K-1,
$$

$$
\mathbf Z=\tilde{\mathbf H}^{(K-1)}(\mathbf W^{(K)})^{\mathsf T}.
$$

Архитектура задаётся `layer_sizes=[d, m_0, ..., m_(K-1), C]`; активации задаются отдельно для каждого скрытого слоя. Смещение хранится в первом столбце каждой матрицы весов.

Для перехода `fan_in → fan_out` создаётся матрица `(fan_out, fan_in + 1)`. Первый столбец содержит веса постоянной координаты: при инициализации они равны нулю, а затем обучаются вместе с остальными весами. Срез `[:, 1:]` получает разные случайные веса. ReLU-слой использует He-масштаб `sqrt(2 / fan_in)`; tanh-слой и выходной линейный слой — Xavier-масштаб `sqrt(2 / (fan_in + fan_out))`. Фиксированный `seed` делает весь набор разных начальных весов воспроизводимым.

На входе слоя $\mathbf H^{(\ell-1)}\in\mathbb R^{B\times m_{\ell-1}}$ является матрицей значений предыдущего уровня. Её строка относится к одному объекту, а столбец — к одному входу слоя. Матрица $\tilde{\mathbf H}^{(\ell-1)}=[\mathbf1_B\ \mathbf H^{(\ell-1)}]$ сохраняет $B$ строк и получает дополнительный первый столбец.

Каждый слой строит свежую цепочку матриц $\mathbf H\to\tilde{\mathbf H}\to\tilde{\mathbf H}\mathbf W^{\mathsf T}\to\varphi(\cdot)$. Объекты весов при этом остаются общими параметрами.

**Действие.** Перенесите общие классы модели, истории и checkpoint, а также функцию графиков. Проследите цикл по матрицам весов и методы `fit`, `evaluate`, `parameters`, `predict_proba`, `predict`. Их применение проверяется следующими ячейками.

**Контроль.** Число скрытых слоёв определяется только длиной `layer_sizes`; код не содержит отдельной ветви для архитектуры Wine.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: mlp_classifier
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 12. Произвольное число слоёв и случай одного объекта

Заполняется блок `mlp_forward_shapes`.

Один класс обязан поддерживать `K=0`, один и несколько скрытых слоёв, разные ширины, ReLU и tanh.

Если batch содержит только объект $i$, то $\mathcal I=\{i\}$ и равенство записывается без потери индекса:

$$
\mathbf H_{\{i\}}^{(-1)}
=\mathbf X_{\{i\}}
=\mathbf x_i^{\mathsf T}
\in\mathbb R^{1\times d}.
$$

Код не получает одномерный массив формы `(d,)`: он получает двумерную матрицу формы `(1, d)`.

**Действие.** Запустите один класс на моделях с `K=0`, `K=1`, `K=2` и на batch из одного объекта.

**Контроль.** Все выходы имеют форму `(B, C)`, а число параметров совпадает с суммой размеров всех матриц весов.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: mlp_forward_shapes
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 13. Mini-batch SGD

Заполняется блок `minibatch_iterator`.

Эпоха — один проход по всем train-объектам. Mini-batch — очередная матрица строк внутри эпохи; после неё выполняется одно обновление весов.

При `B=n` получается full-batch, при `B=1` — SGD, между ними — mini-batch. Последний неполный batch не отбрасывается.

**Действие.** Реализуйте перемешивание индексов и срезы длины `batch_size`.

**Контроль.** Для `125` объектов размеры `125`, `16`, `1` дают соответственно `1`, `8`, `125` обновлений; при `B=16` последний batch содержит `13` объектов.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: minibatch_iterator
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Ячейка 14. Один end-to-end backward

Заполняется блок `single_batch_backward`.

Градиент проходит от CCE через всю маленькую сеть. Сначала fused loss создаёт

$$
\mathbf G_{\mathbf Z}=\frac1B(\mathbf P-\mathbf Y).
$$

Выходной линейный ответ обозначим также $\mathbf R^{(K)}=\mathbf Z$:
у выходного слоя нет отдельной скрытой активации, его логиты поступают прямо
в softmax–CCE. Для слоёв $\ell=K,K-1,\ldots,0$ callbacks получают

$$
\nabla_{\mathbf W^{(\ell)}}L_B
=
(\mathbf G_{\mathbf R^{(\ell)}})^{\mathsf T}
\tilde{\mathbf H}^{(\ell-1)},
$$

а для скрытого слоя дополнительно применяют производную его активации. Один параметр проверяется центральной разностью: два новых forward при $w+\varepsilon$ и $w-\varepsilon$ независимо оценивают ту же производную итоговой потери.

**Действие.** Выполните один backward маленькой MLP, проверьте формы всех градиентов и один вес первой матрицы центральной разностью.

**Контроль.** Каждый параметр получает конечный градиент своей формы; аналитическая и численная производные совпадают.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: single_batch_backward
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.

## Разбор полного примера Dry Bean

К этому моменту вы заполнили первые 14 ячеек Wine по листингам PDF. У вас уже
есть подготовленные данные, `MatrixValue`, локальные проверки, общий класс MLP,
итератор batch и проверенный градиент полной сети. Теперь откройте выполненный
`dry_bean_mlp_full_example.ipynb`: сохранённые результаты позволяют проследить
весь проект, включая те действия, которые ещё предстоит выполнить для Wine.

Dry Bean содержит $13\,611$ объектов, 16 числовых признаков и 7 классов.
Изучайте notebook в отдельном ядре, чтобы его переменные и обученные объекты не
заменили уже созданные данные Wine. Если хотите воспроизвести результаты,
выполните весь пример сверху вниз в этом отдельном ядре.

### Все ячейки образца и их соответствие вашей работе

Число в первом столбце — номер исполнимой ячейки. Там, где названия различаются,
стрелка показывает направление адаптации **Dry Bean $\to$ Wine**. Одинаковые
названия означают одинаковую роль; зависящие от данных проверки всё равно
нужно сверить с последним столбцом.

| № | Ячейка образца $\to$ ячейка Wine | Что сопоставить или адаптировать |
|----|----------------------------------------------|----------------------------------------------|
| 1 | `imports_and_helpers` | Импорты, численный softmax и вывод результатов совпадают. |
| 2 | `load_dry_bean` $\to$ `load_wine` | CSV, порядок признаков, число строк, столбцов, классов и их частоты. |
| 3 | `split_and_standardize` | Общий `Standardizer` совпадает; алгоритм квот и размеры частей различаются. |
| 4 | `matrix_value` | Полный класс и его правила не зависят от датасета. |
| 5 | `transpose_trace` | Та же маленькая матрица и тот же seed; численные результаты совпадают. |
| 6 | `matmul_trace` | Те же два градиента и проверка общего родителя. |
| 7 | `augment_trace` | Постоянный столбец и удаление его градиента работают одинаково. |
| 8 | `activation_trace` | Те же локальные проверки ReLU и tanh. |
| 9 | `softmax_cross_entropy_trace` | Общая loss поддерживает любое допустимое число классов; маленький пример тот же. |
| 10 | `matrix_finite_difference` | Проверки matmul и логита на искусственных матрицах совпадают. |
| 11 | `mlp_classifier` | Общие классы, история, checkpoint и функция графиков совпадают. |
| 12 | `mlp_forward_shapes` | Маленькие диагностические архитектуры остаются прежними. |
| 13 | `minibatch_iterator` | Итератор совпадает; изменить размер train, batch и ожидаемый остаток. |
| 14 | `single_batch_backward` | Та же сеть для проверки градиента; это не обученная модель датасета. |
| 15 | `train_dry_bean` $\to$ `train_wine` | Создать новую Wine-модель: архитектура, число параметров, epochs, batch, имена объектов и checkpoint. |
| 16 | `plot_dry_bean_training_history` $\to$ `plot_wine_training_history` | Передать историю Wine, заменить заголовок и имена переменных; правило выбора эпохи сохраняется. |
| 17 | `evaluate_dry_bean` $\to$ `evaluate_wine` | Применить критерии Wine и получить confusion matrix $3\times3$ с суммой 26. |
| 18 | `serialize_model` | Общий класс bundle совпадает; передать модель, стандартизатор и метки Wine, изменить имя файла. |

### Почему перенос сохраняет математику

Операции `MatrixValue` используют формы входных массивов. Поэтому они
одинаково вычисляют произведение, транспонирование и локальные градиенты при
разных числах объектов, признаков и классов. Для обеих задач строки batch —
объекты, столбцы входа — признаки, столбцы выхода — классы.

Входные целочисленные метки имеют форму `(B,)`. В математической записи им
соответствует one-hot матрица $\mathbf Y\in\{0,1\}^{B\times C}$.
Код fused loss получает индексы правильных классов и реализует вычитание
one-hot единиц по этим индексам; передавать готовую one-hot матрицу в API не
нужно. Для обоих наборов результат сохраняет формулу
$\mathbf G_{\mathbf Z}=(\mathbf P-\mathbf Y)/B$ и форму $(B,C)$.

| Величина | Dry Bean — образец | Wine — ваша задача |
|---|---|---|
| Объекты / признаки / классы | 13611 / 16 / 7 | 178 / 13 / 3 |
| Архитектура | `16→32→16→7` | `13→16→8→3` |
| Первая матрица весов | $(32,17)$ | $(16,14)$ |
| Вторая матрица весов | $(16,33)$ | $(8,17)$ |
| Выходная матрица весов | $(7,17)$ | $(3,9)$ |
| Число параметров | 1191 | 387 |
| Train / validation / test | 9531 / 2040 / 2040 | 125 / 27 / 26 |
| Эпохи / batch | 80 / 128 | 100 / 16 |
| Обновления за эпоху / последний batch | 75 / 59 объектов | 8 / 13 объектов |

Дополнительный столбец каждой матрицы весов соответствует постоянной единице,
а не новому измеренному признаку. Например, первая матрица Wine имеет 16 строк
нейронов и $13+1=14$ столбцов входов. Активации ReLU и tanh, шаг `0.03`, seed
модели `7`, seed разбиения `42` и seed перемешивания `2026` сохраняются.
Сохранение этих чисел не означает равенства весов или статистик двух задач.

### Что требует отдельной адаптации

В Dry Bean функция `stratified_split_70_15_15` для каждого класса вычисляет
`round(0.15 * class_size)` объектов для validation и столько же для test.
Остаток идёт в train. В Wine функция `stratified_split` сначала фиксирует
размеры частей, затем пропорционально распределяет места методом наибольших
остатков: validation — 9/11/7, test — 9/10/7. Оба алгоритма сохраняют
пропорции классов с точностью до округления, но округляют на разных уровнях.
Если применить к Wine округление из Dry Bean, получатся части 124/27/27,
а не требуемые 125/27/26. Поэтому функцию разбиения Wine сохраняйте из его
листинга; фиксированные размеры не означают равные квоты разных классов.

Внутри обучения классы всегда представлены целыми индексами: `0…6` в
Dry Bean и `0…2` в Wine. При выдаче прогноза `InferenceBundle` сопоставляет
индекс столбца вероятности с `class_labels`. В Dry Bean это названия сортов,
в Wine — числа 0, 1 и 2. Общий класс сохраняет тип меток, поэтому при переносе
меняется список меток, а не метод вычисления вероятностей.

Переносите определения и последовательность вызовов. Wine-модель создаётся
заново и обучается на Wine; её `Standardizer` уже вычислил статистики Wine-train
на этапе 3. Для неё создаётся отдельный `BestValidationCheckpoint` с файлом
`wine_best_checkpoint.npz`, а итоговый bundle сохраняется в
`wine_mlp_model.npz`. Веса и train-статистики Dry Bean для этих объектов не
используются.

**Действие студента.** Просмотрите все 18 ячеек образца и их сохранённые
результаты по таблице. Для ячеек 1–14 объясните, что уже реализовано в вашем
Wine-notebook. Затем вернитесь к нему и заполните ячейки 15–18, адаптируя
прикладной код Dry Bean по полным Wine-листингам ниже. Выполняйте их в порядке
`train_wine → plot_wine_training_history → evaluate_wine → serialize_model`.

**Контроль понимания.** До запуска обучения объясните формы трёх матриц Wine,
почему его метки имеют форму `(B,)`, откуда берутся 8 обновлений за эпоху и
почему минимум validation loss выбирает checkpoint до test-оценки.

## Ячейка 15. Классическое обучение эпохами

Заполняется блок `train_wine`.

Для каждого batch соблюдается порядок: свежий forward → loss → `backward()` → $\mathbf W\leftarrow\mathbf W-\eta\nabla_{\mathbf W}L_B$. Архитектура Wine: `13→16→8→3`, активации ReLU и tanh. Следующий batch строит новый DAG уже после обновления параметров.

После каждой эпохи текущие веса оцениваются на одной и той же validation-
матрице без `backward()` и update. Объект `BestValidationCheckpoint` сохраняет
веса эпохи только при уменьшении validation loss. Все 100 эпох выполняются, а
после них студент явно восстанавливает выбранное validation-состояние.

**Действие.** Адаптируйте `train_dry_bean` по Wine-листингу: создайте новую модель `[13, 16, 8, 3]` с 387 параметрами и отдельный checkpoint. Запустите `100` эпох с `batch_size=16`, `learning_rate=0.03` и seed перемешивания `2026`. Используйте подготовленные данные Wine; готовые веса и стандартизатор Dry Bean не переносятся.

**Контроль.** История содержит `100` конечных значений, а последняя train loss меньше первой. Порог loss `< 0.03` и train accuracy `1.0` проверяются в ячейке 17 Evaluation после построения графиков.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: train_wine
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.
# Сопоставьте прикладную ячейку Dry Bean с полным Wine-листингом PDF.

## Ячейка 16. Кривые обучения: train, validation и выбранная эпоха

Заполняется блок `plot_wine_training_history`.

Train-кривые показывают подгонку к объектам, по которым вычислялись градиенты. Validation-кривые показывают качество на фиксированной части данных, которая не участвует в `backward()`. Вертикальная линия отмечает `best_epoch + 1`: checkpoint выбран по минимуму validation loss, а не по последней эпохе.

Test не является третьей кривой обучения. Он не показан на графике и будет использован только один раз на следующем этапе как независимая итоговая оценка.

**Действие.** Адаптируйте вызов `plot_dry_bean_training_history`: используйте `wine_history` и заголовок Wine. Постройте две панели после восстановления checkpoint: loss и accuracy. Убедитесь, что на каждой есть train, validation и линия лучшей validation-эпохи.

**Контроль.** На каждой панели ровно три линии; ось X начинается с эпохи `1`, а test-метрики отсутствуют.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: plot_wine_training_history
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.
# Сопоставьте прикладную ячейку Dry Bean с полным Wine-листингом PDF.

## Ячейка 17. Evaluation

Заполняется блок `evaluate_wine`.

Train показывает подгонку, фиксированная validation выбирает лучший checkpoint,
а test играет роль закрытого итогового экзамена. Test не передаётся в `fit()` и
не влияет на эпоху, веса или гиперпараметры. Confusion matrix хранит правильные
классы по строкам и прогнозы по столбцам.

**Действие.** Адаптируйте `evaluate_dry_bean`: передайте Wine-модель и задайте 3 класса. Вычислите CCE, accuracy и confusion matrix отдельно для трёх частей, не вызывая `backward()` и не меняя веса. Test оценивается только здесь.

**Контроль.** Validation и test accuracy не ниже `0.95`, test CCE меньше `0.2`, confusion matrix имеет форму `(3, 3)` и сумму `26`.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: evaluate_wine
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.
# Сопоставьте прикладную ячейку Dry Bean с полным Wine-листингом PDF.

## Ячейка 18. Deployment: сериализация

Заполняется блок `serialize_model`.

Объект `InferenceBundle` содержит модель, `Standardizer`, порядок признаков и
метки классов. Файл `.npz` сохраняет это состояние без произвольных Python-
объектов. Проверка начинается с отдельного batch сырых признаков, не используемого
для нового выбора модели: загруженные mean/std воспроизводят стандартизацию, после чего новая модель без повторного обучения должна получить
$\mathbf P_{\mathrm{loaded}}\approx\mathbf P_{\mathrm{trained}}$.

**Действие.** Перенесите общий `InferenceBundle`, затем передайте Wine-модель, её стандартизатор, имена признаков и метки `[0, 1, 2]`. Сохраните `wine_mlp_model.npz`, загрузите его в новый объект и подайте отдельный batch сырых признаков. В Dry Bean bundle возвращает названия сортов, а здесь — числа; класс сохраняет тип переданных меток.

**Контроль.** Загруженная модель воспроизводит все вероятности и классы обученной модели без повторного обучения.

In [ ]:
# MATRIX_MLP_AUTOGRAD_ESSENTIALS_CELL: serialize_model
# Перенесите сюда полностью соответствующий блок из выданного текста лабораторной работы.
# Сохраните Google-style docstring, комментарии, проверки и порядок строк.
# Сопоставьте прикладную ячейку Dry Bean с полным Wine-листингом PDF.

## Итог

Вы реализовали матричный reverse-mode, проверили локальные и end-to-end градиенты, обучили произвольную MLP mini-batch SGD, оценили её на независимых данных и сохранили полный inference-bundle.

Итоговая причинная цепочка: матричные операции создают DAG → callbacks реализуют локальные правила → обратный топологический обход получает градиенты всех весов → mini-batch SGD обновляет параметры → независимые validation и test измеряют обобщение → сериализация сохраняет воспроизводимый inference.